# Ingestão Bronze - Squad 3

Este notebook realiza a ingestão das tabelas atribuídas ao Squad 3:

- physical_lojas
- physical_produtos_pereciveis

### Arquitetura

Origem:
- Container `raw`
- Formato de origem: CSV

Destino:
- Container `squad3`
- Camada `bronze`
- Formato: Delta

### Estrutura existente no ADLS2

As estruturas abaixo já foram previamente criadas no container `squad3`:

- `bronze/physical_lojas`
- `bronze/physical_produtos_pereciveis`
- `controle/bronze_processados`

Por esse motivo, este notebook não realiza criação ou exclusão de diretórios.

### Auditoria

São adicionadas as colunas:

- `bronze_ingested_at`: data/hora da ingestão
- `bronze_source_file`: arquivo de origem

### Particionamento

As tabelas Bronze são particionadas por:

- `ano`
- `mes`

Ambos derivados de `bronze_ingested_at`, ou seja, da data de ingestão.

### Frequência

A ingestão será executada semanalmente, às segundas-feiras.

O controle de processamento utiliza o caminho do arquivo e sua data de última
modificação para impedir o reprocessamento da mesma versão do arquivo.

In [0]:
%python

from pyspark.sql.functions import (
    current_timestamp,
    col,
    year,
    month
)

from pyspark.sql import Row
from datetime import datetime

In [0]:
%python

tenant_id = dbutils.secrets.get(
    scope="squad3-secrets",
    key="tenant-id"
)

client_id = dbutils.secrets.get(
    scope="squad3-secrets",
    key="client-id"
)

client_secret = dbutils.secrets.get(
    scope="squad3-secrets",
    key="client-secret"
)

print("Credenciais carregadas com segurança.")

In [0]:
%python

config = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    "fs.azure.account.oauth2.client.id":
        client_id,

    "fs.azure.account.oauth2.client.secret":
        client_secret,

    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Configuração do ADLS2 carregada.")

In [0]:
%python

# ============================================================
# RAW
# ============================================================

caminho_raw_lojas = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_lojas.csv"
)

caminho_raw_produtos = (
    "abfss://raw@internshipdatalake.dfs.core.windows.net/"
    "batch-data/physical_produtos_pereciveis.csv"
)


# ============================================================
# BRONZE
# Estrutura já existente no container squad3
# ============================================================

caminho_bronze_lojas = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
    "bronze/physical_lojas"
)

caminho_bronze_produtos = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
    "bronze/physical_produtos_pereciveis"
)


# ============================================================
# CONTROLE DE PROCESSAMENTO
# Estrutura já existente no container squad3
# ============================================================

caminho_controle = (
    "abfss://squad3@internshipdatalake.dfs.core.windows.net/"
    "controle/bronze_processados"
)

In [0]:
%python

def processar_arquivo(
    caminho_raw,
    caminho_bronze,
    tabela_destino
):

    print(f"\nIniciando: {tabela_destino}")

    # ========================================================
    # 1. Identifica a versão atual do arquivo na Raw
    # ========================================================

    metadata = (
        spark.read
            .format("binaryFile")
            .options(**config)
            .load(caminho_raw)
            .select(
                "path",
                "modificationTime"
            )
            .first()
    )

    arquivo = metadata["path"]
    modificacao = metadata["modificationTime"]


    # ========================================================
    # 2. Consulta o histórico de processamento
    # ========================================================

    df_controle = (
        spark.read
            .format("delta")
            .options(**config)
            .load(caminho_controle)
    )


    # ========================================================
    # 3. Verifica se esta versão já foi processada
    # ========================================================

    ja_processado = (
        df_controle
            .filter(
                (col("bronze_source_file") == arquivo)
                &
                (
                    col("source_modification_time")
                    == modificacao
                )
            )
            .limit(1)
            .count() > 0
    )


    if ja_processado:

        print(
            f"{tabela_destino}: "
            "versão atual do arquivo já processada."
        )

        return


    # ========================================================
    # 4. Lê o CSV da Raw
    # ========================================================

    df = (
        spark.read
            .format("csv")
            .options(**config)
            .option("header", "true")
            .option("inferSchema", "true")
            .load(caminho_raw)
    )


    # ========================================================
    # 5. Adiciona auditoria
    #
    # ano e mes são derivados da DATA DE INGESTÃO.
    # Essas colunas serão utilizadas no particionamento.
    # ========================================================

    df_bronze = (
        df
            .withColumn(
                "bronze_ingested_at",
                current_timestamp()
            )
            .withColumn(
                "bronze_source_file",
                col("_metadata.file_path")
            )
            .withColumn(
                "ano",
                year("bronze_ingested_at")
            )
            .withColumn(
                "mes",
                month("bronze_ingested_at")
            )
    )


    qtd_registros = df.count()


    # ========================================================
    # 6. Grava na Bronze
    #
    # Formato: Delta
    # Modo: Append
    # Particionamento: ano/mes da data de ingestão
    # ========================================================

    (
        df_bronze.write
            .format("delta")
            .mode("append")
            .options(**config)
            .partitionBy(
                "ano",
                "mes"
            )
            .save(caminho_bronze)
    )


    # ========================================================
    # 7. Registra a execução no controle
    # ========================================================

    df_registro = spark.createDataFrame([
        Row(
            bronze_source_file=arquivo,
            source_modification_time=modificacao,
            tabela_destino=tabela_destino,
            bronze_processed_at=datetime.now(),
            qtd_registros=qtd_registros
        )
    ])


    (
        df_registro.write
            .format("delta")
            .mode("append")
            .options(**config)
            .save(caminho_controle)
    )


    print(
        f"{tabela_destino}: "
        f"{qtd_registros} registros processados com sucesso."
    )

In [0]:
%python

processar_arquivo(
    caminho_raw_lojas,
    caminho_bronze_lojas,
    "physical_lojas"
)

In [0]:
%python

processar_arquivo(
    caminho_raw_produtos,
    caminho_bronze_produtos,
    "physical_produtos_pereciveis"
)

In [0]:
%python

df_acompanhamento = (
    spark.read
        .format("delta")
        .options(**config)
        .load(caminho_controle)
        .select(
            "tabela_destino",
            "bronze_source_file",
            "source_modification_time",
            "bronze_processed_at",
            "qtd_registros"
        )
        .orderBy(
            col("bronze_processed_at").desc()
        )
)

display(df_acompanhamento)

In [0]:
%python

df_lojas_check = (
    spark.read
        .format("delta")
        .options(**config)
        .load(caminho_bronze_lojas)
)

df_produtos_check = (
    spark.read
        .format("delta")
        .options(**config)
        .load(caminho_bronze_produtos)
)


print("=== PHYSICAL LOJAS ===")

print(
    "Total de registros:",
    df_lojas_check.count()
)

df_lojas_check \
    .groupBy("ano", "mes") \
    .count() \
    .orderBy("ano", "mes") \
    .show()


print("=== PHYSICAL PRODUTOS PERECÍVEIS ===")

print(
    "Total de registros:",
    df_produtos_check.count()
)

df_produtos_check \
    .groupBy("ano", "mes") \
    .count() \
    .orderBy("ano", "mes") \
    .show()